# Lab 09-14 · Optoelectronic Devices

Companion to chapter [09-14](../../chapters/09-14-optoelectronic-devices.md). Handout: [README.md](README.md).

In [1]:
import math
import numpy as np
from scipy.optimize import brentq, minimize_scalar
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
VT = 1.380649e-23 * 300 / Q
C_LIGHT = 2.99792458e8
rng = np.random.default_rng(914)

## Part 1 — Solar cell

$J = J_{sc} - J_0(e^{(V + JR_s)/V_T} - 1)$, with $J_0$ = 10⁻¹³ A/cm².

*Predict first:* doubling the light: how much does $V_{oc}$ rise, and does efficiency change?

In [2]:
J0 = 1e-13
def cell(jsc, rs=0.0, pin=0.1):
    voc = VT * math.log(jsc / J0 + 1)
    j_of_v = lambda v: brentq(lambda j: jsc - J0 * (math.exp((v + j * rs) / VT) - 1) - j, -1, jsc * 1.01)
    r = minimize_scalar(lambda v: -v * j_of_v(v), bounds=(0, voc), method="bounded", options={"xatol": 1e-12})
    vmp, jmp = r.x, j_of_v(r.x)
    return voc, vmp, jmp, vmp * jmp / (voc * jsc), 100 * vmp * jmp / pin
voc1, vmp1, jmp1, ff1, eta1 = cell(40e-3)
voc2, vmp2, jmp2, ff2, eta2 = cell(80e-3, pin=0.2)
print(f"1 sun: Voc {voc1:.4f} V, Vmp {vmp1:.4f} V, FF {ff1:.3f}, eta {eta1:.1f}%; Rload {vmp1/jmp1:.2f} ohm for 1 cm2")
print(f"2 suns: Voc {voc2:.4f} V (+{1e3*(voc2-voc1):.1f} mV), eta {eta2:.1f}%")
assert abs(voc1 - 0.6906) < 5e-5 and abs(eta1 - 23.3) < 0.05 and abs(1e3 * (voc2 - voc1) - 17.9) < 0.05
assert abs(eta2 - 24.0) < 0.05 and abs(vmp1 / jmp1 - 15.84) < 0.005
for rs in (0.0, 0.5, 1.0, 2.0):
    print(f"Rs = {rs} ohm cm2: FF {cell(40e-3, rs)[3]:.3f}, eta {cell(40e-3, rs)[4]:.1f}%")
v = np.linspace(0, voc1, 300)
j = 40e-3 - J0 * (np.exp(v / VT) - 1)
fig, ax = plt.subplots()
ax.plot(v, j * 1e3, label="J (mA/cm2)"); ax.plot(v, v * j * 1e3, label="P (mW/cm2)")
ax.set_xlabel("V (V)"); ax.legend(); plt.show()

1 sun: Voc 0.6906 V, Vmp 0.6079 V, FF 0.844, eta 23.3%; Rload 15.84 ohm for 1 cm2
2 suns: Voc 0.7085 V (+17.9 mV), eta 24.0%
Rs = 0.0 ohm cm2: FF 0.844, eta 23.3%
Rs = 0.5 ohm cm2: FF 0.818, eta 22.6%
Rs = 1.0 ohm cm2: FF 0.791, eta 21.9%
Rs = 2.0 ohm cm2: FF 0.739, eta 20.4%


## Part 2 — Photon transfer

Each pixel collects a Poisson number of electrons, then gains Gaussian read noise of 2 e⁻ rms.

*Predict first:* the slope of noise against signal on log–log axes in the shot-noise region?

In [3]:
signals = np.logspace(0, 3.8, 20)
noise = []
for s in signals:
    frames = rng.poisson(s, 20000) + rng.normal(0, 2.0, 20000)
    noise.append(frames.std())
noise = np.array(noise)
hi = signals > 300
slope = np.polyfit(np.log(signals[hi]), np.log(noise[hi]), 1)[0]
print(f"shot-noise slope {slope:.3f}; noise at the darkest level {noise[0]:.2f} e- (read noise + shot)")
assert abs(slope - 0.5) < 0.02
plt.loglog(signals, noise, "o-"); plt.xlabel("signal (e-)"); plt.ylabel("noise (e- rms)"); plt.show()

shot-noise slope 0.499; noise at the darkest level 2.23 e- (read noise + shot)


## Part 3 — SPAD time of flight

Target at 5 m; 100 ps rms jitter; 0.1 detected signal photons per pulse; ambient detections uniform over a 100 ns window at a mean of 0.2 per pulse; 20 ps bins.

*Predict first:* how many pulses until the peak stands out?

In [4]:
t_true = 2 * 5.0 / C_LIGHT
def histogram(pulses):
    n_sig = rng.poisson(0.1 * pulses)
    n_amb = rng.poisson(0.2 * pulses)
    times = np.concatenate([rng.normal(t_true, 100e-12, n_sig), rng.uniform(0, 100e-9, n_amb)])
    return np.histogram(times, bins=np.arange(0, 100e-9 + 20e-12, 20e-12))
counts, edges = histogram(2000)
peak = edges[np.argmax(counts)] + 10e-12
print(f"peak at {peak*1e9:.2f} ns -> {C_LIGHT*peak/2:.3f} m (true {t_true*1e9:.2f} ns)")
assert abs(peak - t_true) < 200e-12
plt.plot(edges[:-1] * 1e9, counts); plt.xlabel("time (ns)"); plt.ylabel("counts"); plt.show()

peak at 33.33 ns -> 4.996 m (true 33.36 ns)


## Part 4 — LED extraction

Photons start inside a GaAs chip ($n$ = 3.30) in random directions; count those that reach the top face within the critical angle (single pass), and weight by the Fresnel transmission at their angle (unpolarized average).

*Predict first:* what fraction escapes the top face?

In [5]:
n = math.sqrt(10.89)
N = 400000
u = rng.uniform(-1, 1, N)              # cos(theta) uniform for isotropic directions
up = u[u > 0]
theta_c = math.asin(1 / n)
in_cone = up > math.cos(theta_c)
frac = in_cone.sum() / N
print(f"escape-cone fraction (single pass) {frac:.4f}; formula {(1 - math.cos(theta_c)) / 2:.4f}")
assert abs(frac / ((1 - math.cos(theta_c)) / 2) - 1) < 0.05
# Fresnel transmission inside the cone (average of s and p)
ci = up[in_cone]; si = np.sqrt(1 - ci**2); st = n * si; ct = np.sqrt(1 - st**2)
rs = ((n * ci - ct) / (n * ci + ct))**2; rp = ((ci - n * ct) / (ci + n * ct))**2
print(f"with Fresnel loss: {np.sum(1 - (rs + rp) / 2) / N:.4f} of all photons escape the top face")

escape-cone fraction (single pass) 0.0236; formula 0.0235
with Fresnel loss: 0.0164 of all photons escape the top face
